# VinDr-Mammo: Binary BI-RADS Classification — v7 (Late Fusion)

### Perubahan dari v6:
- **Arsitektur**: CrossViewAttention (mid-fusion) diganti dengan **Late Fusion** (simple concatenation)
- **SwinBiRADS_LateFusion**: CC dan MLO diproses paralel oleh shared-weight encoder → concat → MLP classifier
- **Classifier input**: 2×D (1536 dim) instead of D (768 dim)
- **Rationale**: Cross-View Attention pada global pooled vector (1D) tidak memanfaatkan spatial context; late fusion lebih transparan dan stabil


## 1. Install & Import

In [ ]:
!pip install timm -q
!pip install scikit-learn -q

In [ ]:
import os, random, math
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T

import timm
from sklearn.metrics import (
    classification_report, roc_auc_score, cohen_kappa_score, confusion_matrix,
    f1_score, accuracy_score, recall_score, precision_score,
    roc_curve, precision_recall_curve, average_precision_score
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")

## 2. Configuration

In [ ]:
class CFG:
    DATA_DIR          = '/kaggle/input/datasets/vickyoktafrian/dataset-preprocessed/dataset_preprocessed/dataset_preprocessed'
    CSV_PATH          = '/kaggle/input/datasets/vickyoktafrian/dataset-preprocessed/breast-level_annotations.csv'
    OUTPUT_DIR        = '/kaggle/working/outputs'

    # Gunakan file .pth yang formatnya cocok dengan timm (162/171 loaded)
    MODEL_NAME        = 'swin_tiny_patch4_window7_224'
    PRETRAINED_WEIGHT = '/kaggle/input/datasets/vickyoktafrian/model-mv-swin-t/swin_tiny_patch4_window7_224.pth'
    IMG_SIZE          = 224
    NUM_CLASSES       = 1
    NUM_HEADS         = 8
    DROPOUT           = 0.3

    POSITIVE_BIRADS   = [4, 5]

    EPOCHS            = 100
    BATCH_SIZE        = 16
    NUM_WORKERS       = 4

    WARMUP_EPOCHS     = 5
    LR_BACKBONE       = 1e-5
    LR_HEAD           = 1e-4       # diturunkan dari 3e-4 untuk stabilitas
    WEIGHT_DECAY      = 1e-4
    PATIENCE          = 15

    # KEY FIX: sampler moderat + pos_weight rendah
    # Sampler target 20% positif per batch (dari aslinya 5%)
    # pos_weight 2.0 saja karena sampler sudah bantu balance
    POS_WEIGHT        = 2.0
    SAMPLER_POS_RATIO = 0.20       # target % positif per batch

    # Output bias = log(0.05/0.95)
    OUTPUT_BIAS_INIT  = math.log(0.05 / 0.95)

    USE_MULTI_GPU     = True

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device    : {DEVICE}")
print(f"POS_WEIGHT: {CFG.POS_WEIGHT}")
print(f"Sampler   : {CFG.SAMPLER_POS_RATIO*100:.0f}% positif per batch")
print(f"Bias init : {CFG.OUTPUT_BIAS_INIT:.3f} → sigmoid = {1/(1+math.exp(-CFG.OUTPUT_BIAS_INIT)):.4f}")

## 3. Data

In [ ]:
df = pd.read_csv(CFG.CSV_PATH)

def parse_birads(val):
    if pd.isna(val): return None
    digits = ''.join(filter(str.isdigit, str(val).strip()))
    return int(digits) if digits else None

df['birads_num'] = df['breast_birads'].apply(parse_birads)
df = df.dropna(subset=['birads_num'])
df['birads_num'] = df['birads_num'].astype(int)
df['label'] = df['birads_num'].apply(lambda x: 1 if x in CFG.POSITIVE_BIRADS else 0)

n_pos = df['label'].sum(); n_neg = (df['label']==0).sum()
print(f"Negatif: {n_neg} | Positif: {n_pos} | Ratio: {n_neg/n_pos:.1f}:1")

def build_pairs(df):
    pairs = []
    for (sid, lat), grp in df.groupby(['study_id','laterality']):
        cc  = grp[grp['view_position']=='CC']
        mlo = grp[grp['view_position']=='MLO']
        if len(cc)==0 or len(mlo)==0: continue
        pairs.append({
            'study_id':     sid,
            'laterality':   lat,
            'cc_image_id':  cc.iloc[0]['image_id'],
            'mlo_image_id': mlo.iloc[0]['image_id'],
            'label':        cc.iloc[0]['label'],
            'birads':       cc.iloc[0]['birads_num'],
            'split':        cc.iloc[0]['split']
        })
    return pd.DataFrame(pairs)

pairs_df = build_pairs(df)
train_df = pairs_df[pairs_df['split']=='training'].reset_index(drop=True)
val_df   = pairs_df[pairs_df['split']=='test'].reset_index(drop=True)

if len(val_df)==0:
    from sklearn.model_selection import train_test_split
    train_df, val_df = train_test_split(pairs_df, test_size=0.2,
                                        stratify=pairs_df['label'], random_state=42)
    train_df = train_df.reset_index(drop=True)
    val_df   = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Pos: {int(train_df['label'].sum())} ({train_df['label'].mean():.2%})")
print(f"Val  : {len(val_df)}   | Pos: {int(val_df['label'].sum())} ({val_df['label'].mean():.2%})")
print(f"\n[SANITY] CC==MLO image_id: {(train_df['cc_image_id']==train_df['mlo_image_id']).sum()} (harus 0)")

## 4. Dataset & DataLoader

In [ ]:
def get_image_path(data_dir, study_id, image_id):
    base = Path(data_dir) / str(study_id)
    for ext in ['.png', '.jpg', '.jpeg']:
        p = base / f'{image_id}{ext}'
        if p.exists(): return str(p)
    for ext in ['.png', '.jpg', '.jpeg']:
        p = Path(data_dir) / f'{image_id}{ext}'
        if p.exists(): return str(p)
    return None


def get_transforms(split='train', img_size=224):
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    if split == 'train':
        return T.Compose([
            T.Resize((img_size, img_size)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.2),
            T.RandomRotation(degrees=15),
            T.ColorJitter(brightness=0.3, contrast=0.3),
            T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            T.ToTensor(),
            T.Normalize(mean=mean, std=std),
        ])
    return T.Compose([T.Resize((img_size, img_size)), T.ToTensor(), T.Normalize(mean, std)])


class VinDrDataset(Dataset):
    def __init__(self, df, data_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.data_dir  = data_dir
        self.transform = transform

    def __len__(self): return len(self.df)

    def _load(self, study_id, image_id):
        path = get_image_path(self.data_dir, study_id, image_id)
        if path is None:
            return Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
        return Image.open(path).convert('RGB')

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cc  = self._load(row['study_id'], row['cc_image_id'])
        mlo = self._load(row['study_id'], row['mlo_image_id'])
        # CC dan MLO di-transform INDEPENDEN — tidak pakai seed yang sama
        if self.transform:
            cc  = self.transform(cc)
            mlo = self.transform(mlo)
        return cc, mlo, torch.tensor(row['label'], dtype=torch.float32)


train_dataset = VinDrDataset(train_df, CFG.DATA_DIR, get_transforms('train', CFG.IMG_SIZE))
val_dataset   = VinDrDataset(val_df,   CFG.DATA_DIR, get_transforms('val',   CFG.IMG_SIZE))

# Sampler moderat: target SAMPLER_POS_RATIO% positif per batch
# Tidak 50:50 (terlalu agresif) dan tidak distribusi asli (terlalu jarang positif)
labels_list = train_df['label'].tolist()
n_neg_tr = labels_list.count(0)
n_pos_tr = labels_list.count(1)
r = CFG.SAMPLER_POS_RATIO
w_pos = r / n_pos_tr
w_neg = (1 - r) / n_neg_tr
sample_weights = [w_pos if l == 1 else w_neg for l in labels_list]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset, batch_size=CFG.BATCH_SIZE,
    sampler=sampler, num_workers=CFG.NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False,
    num_workers=CFG.NUM_WORKERS, pin_memory=True
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# Sanity check
cc_b, mlo_b, lbl_b = next(iter(train_loader))
print(f"Batch labels : {lbl_b.tolist()}")
print(f"Pos per batch: {int(lbl_b.sum())}/{len(lbl_b)} (~{lbl_b.mean()*100:.0f}%, target ~{r*100:.0f}%)")
print(f"CC == MLO    : {torch.allclose(cc_b, mlo_b)} ← harus False")

## 5. Model — Late Fusion


In [ ]:
class SwinBiRADS_LateFusion(nn.Module):
    """
    Late-Fusion Swin Transformer for BI-RADS binary classification.

    Architecture:
      CC  ──► Swin Encoder (shared weights) ──► cc_feat  [B, D]
      MLO ──► Swin Encoder (shared weights) ──► mlo_feat [B, D]
                                                     │
                                          torch.cat([cc_feat, mlo_feat], dim=1)  → [B, 2D]
                                                     │
                                               MLP Classifier
                                                     │
                                              logit [B, 1]

    Notes:
      - Encoder weights are SHARED between CC and MLO (weight-tying).
      - No cross-view interaction at the feature level (pure late fusion).
      - Classifier input dim = 2 × encoder.num_features (e.g. 1536 for swin_tiny).
    """

    def __init__(self, cfg):
        super().__init__()

        # ── Shared-weight Swin encoder ─────────────────────────────────────
        # num_classes=0 + global_pool='avg' → output is feature vector [B, D]
        self.encoder = timm.create_model(
            cfg.MODEL_NAME, pretrained=False, num_classes=0, global_pool='avg'
        )
        D = self.encoder.num_features          # e.g. 768 for swin_tiny
        print(f"Encoder dim (D)        : {D}")
        print(f"Classifier input dim   : {D * 2}  (2×D after late-fusion concat)")

        # ── MLP Classifier on concatenated features ────────────────────────
        # Input: [B, 2D]  →  Output: [B, 1] (logit)
        self.classifier = nn.Sequential(
            nn.Linear(D * 2, 512),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT / 2),
            nn.Linear(128, cfg.NUM_CLASSES)
        )

        # ── Weight initialisation ──────────────────────────────────────────
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

        # Bias on last linear → init prob ≈ class prior (e.g. 5 %)
        last_linear = [m for m in self.classifier.modules() if isinstance(m, nn.Linear)][-1]
        nn.init.constant_(last_linear.bias, cfg.OUTPUT_BIAS_INIT)
        init_prob = 1 / (1 + math.exp(-cfg.OUTPUT_BIAS_INIT))
        print(f"Output bias init       : {cfg.OUTPUT_BIAS_INIT:.3f}  →  init prob = {init_prob:.4f}")

    def forward(self, cc: torch.Tensor, mlo: torch.Tensor) -> torch.Tensor:
        """
        Args:
            cc  : Tensor [B, C, H, W] — CC view
            mlo : Tensor [B, C, H, W] — MLO view
        Returns:
            logit : Tensor [B] — raw logit (apply sigmoid for probability)
        """
        cc_feat  = self.encoder(cc)                          # [B, D]
        mlo_feat = self.encoder(mlo)                         # [B, D]
        fused    = torch.cat([cc_feat, mlo_feat], dim=1)     # [B, 2D]
        return self.classifier(fused).squeeze(-1)            # [B]


# ── Instantiate model ──────────────────────────────────────────────────────────
model = SwinBiRADS_LateFusion(CFG)

# ── Load pretrained weights (encoder only) ────────────────────────────────────
ckpt = torch.load(CFG.PRETRAINED_WEIGHT, map_location='cpu')
sd   = ckpt.get('model', ckpt.get('state_dict', ckpt)) if isinstance(ckpt, dict) else ckpt
ms   = model.encoder.state_dict()
filt = {k: v for k, v in sd.items() if k in ms and v.shape == ms[k].shape}
model.encoder.load_state_dict(filt, strict=False)
print(f"Pretrained: {len(filt)}/{len(ms)} layers loaded")
assert len(filt) > 100, f"Terlalu sedikit layer loaded ({len(filt)}) — cek file .pth!"

if CFG.USE_MULTI_GPU and torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(DEVICE)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")


## 6. Loss & Optimizer

In [ ]:
pos_w     = torch.tensor([CFG.POS_WEIGHT], device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
print(f"Loss: BCEWithLogitsLoss(pos_weight={CFG.POS_WEIGHT})")


def get_optimizer(model, cfg, backbone_frozen=True):
    base = model.module if hasattr(model, 'module') else model
    backbone_lr = 0.0 if backbone_frozen else cfg.LR_BACKBONE
    return torch.optim.AdamW([
        {'params': base.encoder.parameters(),    'lr': backbone_lr,  'weight_decay': cfg.WEIGHT_DECAY},
        {'params': base.classifier.parameters(), 'lr': cfg.LR_HEAD,  'weight_decay': cfg.WEIGHT_DECAY},
    ])


def freeze_backbone(model, freeze=True):
    base = model.module if hasattr(model, 'module') else model
    for p in base.encoder.parameters():
        p.requires_grad = not freeze
    print(f"Backbone {'FROZEN' if freeze else 'UNFROZEN'}")


freeze_backbone(model, freeze=True)
optimizer = get_optimizer(model, CFG, backbone_frozen=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG.EPOCHS, eta_min=1e-7
)
print(f"LR head: {CFG.LR_HEAD} | LR backbone: {CFG.LR_BACKBONE} (frozen {CFG.WARMUP_EPOCHS} epoch pertama)")


## 7. Diagnostic Pre-Training

In [ ]:
print("=" * 60)
print("DIAGNOSTIC v6")
print("=" * 60)

cc_b, mlo_b, lbl_b = next(iter(train_loader))
cc_b, mlo_b, lbl_b = cc_b.to(DEVICE), mlo_b.to(DEVICE), lbl_b.to(DEVICE)

# [1] CC vs MLO pixel check
identical = torch.allclose(cc_b, mlo_b)
print(f"\n[1] CC == MLO pixel-identik: {identical}", '❌ BUG!' if identical else '✅ OK')

# [2] Feature check
base = model.module if hasattr(model, 'module') else model
base.eval()
with torch.no_grad():
    f_cc  = base.encoder(cc_b)
    f_mlo = base.encoder(mlo_b)
feat_id   = torch.allclose(f_cc, f_mlo)
feat_diff = (f_cc - f_mlo).abs().mean().item()
print(f"[2] f_cc == f_mlo: {feat_id}", '❌ COLLAPSE!' if feat_id else '✅ OK')
print(f"    Mean diff: {feat_diff:.6f} (harus > 0)")

# [3] Prob awal
base.train()
with torch.no_grad():
    logits = model(cc_b, mlo_b)
    probs  = torch.sigmoid(logits)
print(f"\n[3] Labels: {lbl_b.tolist()}")
print(f"    Probs : {[f'{p:.3f}' for p in probs.cpu().tolist()]}")
print(f"    Mean  : {probs.mean():.4f} (target ~0.05)")

# [4] Gradient & weight update check
base.train()
w_before = base.classifier[0].weight.data.clone()
optimizer.zero_grad()
logits = model(cc_b, mlo_b)
loss   = criterion(logits, lbl_b)
loss.backward()
g_norm = base.classifier[0].weight.grad.norm().item()
optimizer.step()
w_change = (base.classifier[0].weight.data - w_before).abs().max().item()
print(f"\n[4] Loss          : {loss.item():.4f}")
print(f"    Grad norm     : {g_norm:.6f}", '✅' if g_norm > 1e-6 else '❌ NO GRAD')
print(f"    Weight change : {w_change:.8f}", '✅' if w_change > 1e-10 else '❌ NO UPDATE')
model.zero_grad()

# [5] Batch positif
pos_in_batch = int(lbl_b.sum().item())
print(f"\n[5] Positif dalam batch: {pos_in_batch}/{len(lbl_b)}",
      '✅' if pos_in_batch >= 2 else '⚠️ Terlalu sedikit positif')

print("\n" + "=" * 60)
print("Semua ✅ → lanjut training. Ada ❌ → stop dulu.")
print("=" * 60)

## 8. Training

In [ ]:
def find_best_threshold(labels, probs):
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.05, 0.95, 0.01):
        p  = (probs >= t).astype(int)
        f1 = f1_score(labels, p, average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1


def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss = 0
    all_probs, all_labels = [], []

    for cc, mlo, labels in tqdm(loader, desc='Train', leave=False):
        cc, mlo, labels = cc.to(device), mlo.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(cc, mlo)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        all_probs.extend(torch.sigmoid(logits).detach().cpu().float().numpy())
        all_labels.extend(labels.cpu().numpy())

    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    try:    auc = roc_auc_score(labels, probs)
    except: auc = 0.5
    t, f1  = find_best_threshold(labels, probs)
    pos_pm = probs[labels == 1].mean() if (labels == 1).sum() > 0 else 0.0
    neg_pm = probs[labels == 0].mean() if (labels == 0).sum() > 0 else 0.0
    return total_loss / len(loader), auc, f1, pos_pm, neg_pm


@torch.no_grad()
def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_probs, all_labels = [], []

    for cc, mlo, labels in tqdm(loader, desc='Val', leave=False):
        cc, mlo, labels = cc.to(device), mlo.to(device), labels.to(device)
        with torch.cuda.amp.autocast():
            logits = model(cc, mlo)
            loss   = criterion(logits, labels)
        total_loss += loss.item()
        all_probs.extend(torch.sigmoid(logits).detach().cpu().float().numpy())
        all_labels.extend(labels.cpu().numpy())

    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    try:    auc = roc_auc_score(labels, probs)
    except: auc = 0.5
    t, f1  = find_best_threshold(labels, probs)
    pos_pm = probs[labels == 1].mean() if (labels == 1).sum() > 0 else 0.0
    neg_pm = probs[labels == 0].mean() if (labels == 0).sum() > 0 else 0.0
    return total_loss / len(loader), auc, f1, probs, labels, t, pos_pm, neg_pm


print("Training functions defined.")

In [ ]:
scaler  = torch.cuda.amp.GradScaler()
history = {k: [] for k in ['tr_loss', 'vl_loss', 'tr_auc', 'vl_auc', 'tr_f1', 'vl_f1']}

best_auc          = 0.0
patience_count    = 0
best_probs        = None
best_labels_arr   = None
best_thresh_saved = 0.5

print(f"{'Ep':>3} | {'TrLoss':>7} | {'VlAUC':>6} | {'VlF1':>6} | {'pos_pm':>6} | {'neg_pm':>6} | Status")
print('-' * 80)

for epoch in range(1, CFG.EPOCHS + 1):

    if epoch == CFG.WARMUP_EPOCHS + 1:
        freeze_backbone(model, freeze=False)
        optimizer = get_optimizer(model, CFG, backbone_frozen=False)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CFG.EPOCHS - CFG.WARMUP_EPOCHS, eta_min=1e-7
        )
        print(f"  → Backbone unfrozen @ epoch {epoch}")

    tr_loss, tr_auc, tr_f1, tr_pos_pm, tr_neg_pm = train_epoch(
        model, train_loader, optimizer, criterion, DEVICE, scaler
    )
    vl_loss, vl_auc, vl_f1, vl_probs, vl_labels, vl_t, vl_pos_pm, vl_neg_pm = val_epoch(
        model, val_loader, criterion, DEVICE
    )
    scheduler.step()

    history['tr_loss'].append(tr_loss); history['vl_loss'].append(vl_loss)
    history['tr_auc'].append(tr_auc);   history['vl_auc'].append(vl_auc)
    history['tr_f1'].append(tr_f1);     history['vl_f1'].append(vl_f1)

    status = ''
    if vl_auc > best_auc:
        best_auc = vl_auc
        patience_count = 0
        base = model.module if hasattr(model, 'module') else model
        torch.save(base.state_dict(), f'{CFG.OUTPUT_DIR}/best_model.pth')
        best_probs, best_labels_arr = vl_probs, vl_labels
        best_thresh_saved = vl_t
        status = '✓ SAVED'
    else:
        patience_count += 1
        if patience_count >= CFG.PATIENCE:
            print(f"\nEarly stopping @ epoch {epoch}")
            break

    # Tanda ← muncul kalau pos_pm mulai berbeda dari neg_pm
    sep = ' ←' if vl_pos_pm > vl_neg_pm * 1.1 else ''
    print(f"{epoch:>3} | {tr_loss:>7.4f} | {vl_auc:>6.4f} | {vl_f1:>6.4f} | "
          f"{vl_pos_pm:>6.3f} | {vl_neg_pm:>6.3f} | {status}{sep}")

print(f"\n✅ Selesai. Best Val AUC: {best_auc:.4f}")

## 9. Evaluation

In [ ]:
base = model.module if hasattr(model, 'module') else model
base.load_state_dict(torch.load(f'{CFG.OUTPUT_DIR}/best_model.pth', map_location=DEVICE))

_, _, _, best_probs, best_labels_arr, best_thresh_saved, vl_pos_pm, vl_neg_pm = val_epoch(
    model, val_loader, criterion, DEVICE
)

print(f"Val prob: pos_mean={vl_pos_pm:.3f} | neg_mean={vl_neg_pm:.3f}")
print(f"Threshold: {best_thresh_saved:.2f}\n")

# Threshold sweep
print("Threshold sweep:")
for t in np.arange(0.05, 0.96, 0.05):
    p    = (best_probs >= t).astype(int)
    sens = recall_score(best_labels_arr, p, pos_label=1, zero_division=0)
    spec = recall_score(best_labels_arr, p, pos_label=0, zero_division=0)
    f1   = f1_score(best_labels_arr, p, average='macro', zero_division=0)
    prec = precision_score(best_labels_arr, p, pos_label=1, zero_division=0)
    print(f"  t={t:.2f} | F1={f1:.4f} | Sens={sens:.3f} | Spec={spec:.3f} | Prec={prec:.3f}")

final_preds = (best_probs >= best_thresh_saved).astype(int)
auc         = roc_auc_score(best_labels_arr, best_probs)
acc         = accuracy_score(best_labels_arr, final_preds)
f1_macro    = f1_score(best_labels_arr, final_preds, average='macro',    zero_division=0)
f1_weight   = f1_score(best_labels_arr, final_preds, average='weighted', zero_division=0)
kappa       = cohen_kappa_score(best_labels_arr, final_preds)
sensitivity = recall_score(best_labels_arr, final_preds, pos_label=1, zero_division=0)
specificity = recall_score(best_labels_arr, final_preds, pos_label=0, zero_division=0)
precision   = precision_score(best_labels_arr, final_preds, pos_label=1, zero_division=0)

print("\n" + "="*50)
print("       EVALUATION RESULTS (v6)")
print("="*50)
print(f"  AUC-ROC      : {auc:.4f}")
print(f"  Accuracy     : {acc:.4f}")
print(f"  Macro F1     : {f1_macro:.4f}")
print(f"  Sensitivity  : {sensitivity:.4f}")
print(f"  Specificity  : {specificity:.4f}")
print(f"  Precision    : {precision:.4f}")
print(f"  Kappa        : {kappa:.4f}")
print("="*50)
print(classification_report(best_labels_arr, final_preds,
      target_names=['Negatif (1,2,3)', 'Positif (4,5)'], zero_division=0))

## 10. Visualisasi

In [ ]:
fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(2, 3, figure=fig)

ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(history['tr_loss'], label='Train', color='#3498db', lw=2)
ax1.plot(history['vl_loss'], label='Val',   color='#e74c3c', lw=2)
ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(history['tr_auc'], label='Train', color='#3498db', lw=2)
ax2.plot(history['vl_auc'], label='Val',   color='#e74c3c', lw=2)
ax2.axhline(y=best_auc, color='green', ls='--', alpha=0.7, label=f'Best={best_auc:.4f}')
ax2.set_title('AUC-ROC'); ax2.legend(); ax2.grid(alpha=0.3)

ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(history['tr_f1'], label='Train', color='#3498db', lw=2)
ax3.plot(history['vl_f1'], label='Val',   color='#e74c3c', lw=2)
ax3.set_title('Macro F1'); ax3.legend(); ax3.grid(alpha=0.3)

ax4 = fig.add_subplot(gs[1, 0])
cm = confusion_matrix(best_labels_arr, final_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax4,
            xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
ax4.set_title('Confusion Matrix')
ax4.set_xlabel('Predicted'); ax4.set_ylabel('True')

ax5 = fig.add_subplot(gs[1, 1])
fpr, tpr, _ = roc_curve(best_labels_arr, best_probs)
ax5.plot(fpr, tpr, color='#e74c3c', lw=2, label=f'AUC={auc:.4f}')
ax5.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax5.set_title('ROC Curve'); ax5.legend(); ax5.grid(alpha=0.3)
ax5.set_xlabel('FPR'); ax5.set_ylabel('TPR')

ax6 = fig.add_subplot(gs[1, 2])
prec_arr, rec_arr, _ = precision_recall_curve(best_labels_arr, best_probs)
ap = average_precision_score(best_labels_arr, best_probs)
ax6.plot(rec_arr, prec_arr, color='#9b59b6', lw=2, label=f'AP={ap:.4f}')
ax6.set_title('Precision-Recall'); ax6.legend(); ax6.grid(alpha=0.3)
ax6.set_xlabel('Recall'); ax6.set_ylabel('Precision')

plt.tight_layout()
plt.savefig(f'{CFG.OUTPUT_DIR}/results_v7.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot disimpan.")

## 11. Summary

In [ ]:
print("="*60)
print("           FINAL SUMMARY (v7 — Late Fusion)")
print("="*60)
print(f"  Pretrained : swin_tiny_patch4_window7_224.pth (162/171)")
print(f"  Sampler    : {CFG.SAMPLER_POS_RATIO*100:.0f}% positif per batch")
print(f"  POS_WEIGHT : {CFG.POS_WEIGHT}")
print(f"  LR         : backbone={CFG.LR_BACKBONE}, head={CFG.LR_HEAD}")
print(f"  Warmup     : {CFG.WARMUP_EPOCHS} epoch")
print(f"  Threshold  : {best_thresh_saved:.2f}")
print("-"*60)
print(f"  AUC-ROC    : {auc:.4f}")
print(f"  Macro F1   : {f1_macro:.4f}")
print(f"  Sensitivity: {sensitivity:.4f}")
print(f"  Specificity: {specificity:.4f}")
print(f"  Kappa      : {kappa:.4f}")
print("="*60)

pd.DataFrame([{
    'auc': auc, 'accuracy': acc, 'macro_f1': f1_macro, 'weighted_f1': f1_weight,
    'sensitivity': sensitivity, 'specificity': specificity,
    'kappa': kappa, 'precision': precision, 'threshold': best_thresh_saved
}]).to_csv(f'{CFG.OUTPUT_DIR}/metrics_v7.csv', index=False)
print("Saved to metrics_v7.csv")